# Data

Loads the analysis sample and builds, for each configuration, the matrix and block layout that the distance
functions need.

Every configuration uses the same 216 patients: those with complete clinical data and all five image types.
Missing values are not imputed. The loader stops with an error if it does not get exactly 216 patients,
110 components per image type and no missing values, so no analysis can run on a different sample by mistake.

Imports

In [ ]:
import numpy as np
import pandas as pd

### Image columns

The PCA columns of one image type (`cor_PC1 … cor_PC110`), sorted by number (sorting them as text would put PC10 before PC2).

In [ ]:
def image_columns(image_type, df):
    cols = [c for c in df.columns if c.startswith(f"{image_type}_PC")]
    return sorted(cols, key=lambda c: int(c.rsplit("PC", 1)[1]))

### Loading the analysis sample

1. Read the clinical spreadsheet.
2. Remove the `_MR1` ending from the patient IDs (`OAS1_0001_MR1` → `OAS1_0001`), the format used by the image
   files.
3. Drop patients with a missing value in Age, MMSE, eTIV, nWBV, Education, SES, CDR or sex.
4. Code sex as 1 = man, 0 = woman. ASF is not loaded.
5. Join the five PCA files, keeping only patients present in all of them.
6. Sort by patient ID and run the checks.

In [ ]:
def load_analysis_data():
    """The 216 OASIS-1 patients with complete clinical data and all five image types.

    No imputation: patients missing any clinical variable or CDR are dropped.
    """
    clin = pd.read_excel(cfg.CLINICAL_FILE).rename(columns={"ID": "patient_id"})
    clin["patient_id"] = clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
    clin = clin.drop_duplicates(subset="patient_id", keep="first")
    other = [v for v in cfg.CLINICAL_MULTICLASS if v != cfg.SEX]
    clin = clin.dropna(subset=cfg.CLINICAL_QUANT + other + [cfg.CDR, "M/F"])
    clin[cfg.SEX] = (clin["M/F"] == "M").astype(float)
    clin = clin[["patient_id", "M/F", cfg.SEX, cfg.CDR] + cfg.CLINICAL_QUANT + other]

    df = clin
    for t in cfg.IMAGE_TYPES:
        pcs = pd.read_csv(cfg.DATA_DIR / f"pca_{t}.csv")
        df = df.merge(pcs, on="patient_id", how="inner")

    df = df.sort_values("patient_id").reset_index(drop=True).copy()

    if len(df) != cfg.EXPECTED_N:
        raise ValueError(f"Expected {cfg.EXPECTED_N} patients, got {len(df)}")
    for t in cfg.IMAGE_TYPES:
        if len(image_columns(t, df)) != cfg.N_PCS:
            raise ValueError(f"{t}: expected {cfg.N_PCS} PCs")
    if df.isna().any().any():
        raise ValueError("Missing values in the analysis sample")
    return df

### Replacing the image components

Swaps the image components for another set (same patients, same clinical values). Used in `analysis/06_pca_robustness` to test 55, 72 and 163 components.

In [ ]:
def with_image_pcs(df, pcs):
    """Copy of df whose image PCs are replaced by `pcs` (patient_id + '<type>_PC<i>' columns)."""
    old = [c for t in cfg.IMAGE_TYPES for c in image_columns(t, df)]
    out = df.drop(columns=old).merge(pcs, on="patient_id", how="left")
    if len(out) != len(df) or out.isna().any().any():
        raise ValueError("new PCs do not cover every patient")
    return out

### Block columns

Expands `"image:cor"` into the 110 `cor_PC` columns.

In [ ]:
def _block_columns(block_cols, df):
    if isinstance(block_cols, str) and block_cols.startswith("image:"):
        return image_columns(block_cols.split(":", 1)[1], df)
    return list(block_cols)

### Block layout of a configuration

The distance functions need one matrix with the columns in order (quantitative blocks, then binary, then
multiclass), plus `p1` (number of columns in each quantitative block), `d1` (distance used in each of them) and
`p3` (number of multiclass variables, compared with the Hamming distance). For M1:

```
blocks: cor | gfc_sag | gfc_tra | masked_tra | sbj_sag | clinical      | multiclass
p1    = [110, 110, 110, 110, 110, 4]                                   p3 = 3
d1    = [euclidean ×5, robust_mahalanobis]                             d3 = hamming
```

Images and clinical variables are always separate blocks. An empty block (for example from a typo in the
configuration) stops with an error.

In [ ]:
def build_design(df, config):
    """Column-ordered matrix and block layout for a k-medoids configuration.

    Columns are ordered quantitative blocks first, then binary, then multiclass, as robust_mixed_dist expects.
    """
    blocks = [(label, _block_columns(cols, df), dist) for label, cols, dist in config["quant_blocks"]]
    quant_cols = [c for _, cols, _ in blocks for c in cols]
    binary_cols = list(config["binary"])
    multiclass_cols = list(config.get("multiclass", []))
    columns = quant_cols + binary_cols + multiclass_cols

    p1 = [len(cols) for _, cols, _ in blocks]
    if any(n == 0 for n in p1):
        raise ValueError(f"empty block in {[label for label, _, _ in blocks]}")
    d1 = [dist for _, _, dist in blocks]
    return {
        "X": df[columns].to_numpy(dtype=np.float64),
        "columns": columns,
        "block_labels": [label for label, _, _ in blocks] + (["multiclass"] if multiclass_cols else []),
        "p1": p1 if len(p1) > 1 else p1[0],
        "d1": d1 if len(d1) > 1 else d1[0],
        "p2": len(binary_cols),
        "p3": len(multiclass_cols),
        "d2": "sokal",
        "d3": "hamming",
    }

### Input for k-means (C1-b and I2-b)

Age, MMSE, eTIV and nWBV standardised to mean 0 and standard deviation 1 (otherwise eTIV, in cm³, would dominate),
and Education, SES and sex one-hot encoded: one 0/1 column per category, so that k-means only sees whether two
patients share a category, as the Hamming distance does in the other configurations.

For I2-b, the 550 image PCs are used as they are, since they are already on a common scale.

In [ ]:
def build_kmeans_matrix(df, config):
    """k-means input: standardised quantitative variables, one 0/1 column per category, and columns used as they are."""
    parts = []
    if config.get("kmeans_standardise"):
        std = df[config["kmeans_standardise"]].astype(float)
        parts.append((std - std.mean()) / std.std(ddof=0))
    if config.get("kmeans_onehot"):
        parts.append(pd.get_dummies(df[config["kmeans_onehot"]].astype("category"), dtype=float))
    raw = [c for spec in config.get("kmeans_raw", []) for c in _block_columns(spec, df)]
    if raw:
        parts.append(df[raw].astype(float))
    X = pd.concat(parts, axis=1)
    return {"X": X.to_numpy(dtype=np.float64), "columns": list(X.columns)}